# 进阶教程（四）：流式输出与回调机制

> 流式 = 体验（用户等待时间从"总耗时"降到"首 token 时间"）；
> 回调 = 可观测性（token 统计、耗时分析、审计日志）。

## 本讲内容
1. LLM / Chain 的 token 级流式
2. LangGraph 四种 stream_mode 与组合
3. create_agent 的流式
4. 自定义回调 Handler
5. LangSmith 追踪

# 0. 环境准备与运行说明

**前置要求：**
- 根目录 `.env` 已配置 `DEEPSEEK_API_KEY`（本教程用真实 DeepSeek，无本地降级）
- 已安装：`langchain>=1.3`、`langgraph>=1.2`、`langchain-deepseek`、`python-dotenv`
- 使用本地 `bge-small-zh-v1.5` 嵌入的章节首次运行会下载模型（约 100MB，走 hf-mirror 镜像）

**运行说明：**
- 按 cell 顺序执行；除标注外，每个示例消耗少量 API 额度（单次 < 0.01 元量级）
- 本教程面向已学完 `langchain_tutorial/` 与 `langgraph_tutorial/` 基础篇的开发者
- 涉及导入路径的坑（如 `create_agent` 在 `langchain.agents`）已在 FAQ 中汇总

In [1]:

# ========== 0. 初始化（每个 notebook 第一格） ==========
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)

# HF 镜像必须先于任何 langchain/huggingface 导入设置（详见 rag_qa_project FAQ）
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

ROOT = Path.cwd().parent  # advanced_tutorial 的上一级 = 项目根目录
sys.path.insert(0, str(ROOT))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

assert os.getenv("DEEPSEEK_API_KEY"), "请先在根目录 .env 配置 DEEPSEEK_API_KEY"

# 真实 LLM：DeepSeek（本教程要求真实模型）
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat", temperature=0.2)
print("模型就绪:", model.__class__.__name__)


模型就绪: ChatDeepSeek


## 1. LLM token 级流式

`stream()` 返回生成器，逐块产出 AIMessageChunk：

In [2]:

print("打字机效果: ", end="")
for chunk in model.stream("用 50 字解释什么是向量检索"):
    print(chunk.content, end="", flush=True)
print()

打字机效果: 向量检索是将数据转为向量，通过计算向量间距离（如余弦相似度）来快速查找最相似内容的技术，广泛用于搜索引擎和推荐系统。


## 2. LCEL 链的流式

链的 `stream()` 会**穿透**到内部的最小流式单元（这里是模型），
其他环节（prompt 组装、解析）逐条传递，无需改一行代码。

In [3]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = ChatPromptTemplate.from_template("一句话解释：{x}") | model | StrOutputParser()

for token in chain.stream({"x": "checkpointer"}):
    print(token, end="", flush=True)
print()

**Checkpointer** 是 LangGraph 中用于**自动保存和恢复图执行状态**的组件，它让工作流在中断或失败后能从上一步继续运行。


## 3. LangGraph 的 stream_mode

图执行 = 多节点接力，"流什么"有四种选择：

| mode | 流出内容 | 适用 |
|---|---|---|
| `values` | 每步后的**完整状态** | 状态审计 |
| `updates` | 每步的**状态增量**（按节点分组） | 调试、进度展示 |
| `messages` | LLM 的 **token 块**（含元数据） | 前端打字机 |
| `debug` | 全部事件（含任务调度细节） | 深度排障 |

In [6]:

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class WS(TypedDict):
    topic: str
    steps: list

def n1(state: WS):
    return {"steps": [f"大纲：{state['topic']}三部分"]}

def n2(state: WS):
    r = model.invoke(f"按此大纲写一句话简介：{state['steps'][0]}")
    return {"steps": state["steps"] + [f"简介：{r.content}"]}

b = StateGraph(WS)
b.add_node("n1", n1); b.add_node("n2", n2)
b.add_edge(START, "n1"); b.add_edge("n1", "n2"); b.add_edge("n2", END)
demo = b.compile()

print("=== updates 模式：只看增量 ===")
for upd in demo.stream({"topic": "RAG", "steps": []}, stream_mode="updates"):
    for node, delta in upd.items():
        print(f"[{node}] 新增: {list(delta.keys())}")


print("\n=== values 模式：每步完整状态 ===")
for i, v in enumerate(demo.stream({"topic": "RAG", "steps": []}, stream_mode="values")):
    print(f"step{i}: steps={len(v['steps'])}")

=== updates 模式：只看增量 ===
[n1] 新增: ['steps']
[n2] 新增: ['steps']

=== values 模式：每步完整状态 ===
step0: steps=0
step1: steps=1
step2: steps=2


### 多模式组合：进度 + 打字机同时要

传入 mode 列表，每个 chunk 是 `(mode, payload)` 元组——
rag_qa_project 的 `run.py --stream` 正是这种用法：

In [7]:

print("回答: ", end="")
for mode, chunk in demo.stream({"topic": "Agent", "steps": []},
                               stream_mode=["updates", "messages"]):
    if mode == "messages":
        msg, meta = chunk
        content = getattr(msg, "content", "")
        if content and meta.get("langgraph_node"):
            print(content, end="", flush=True)
    elif mode == "updates":
        pass  # 需要时在这里展示节点进度
print()

回答: 根据您的大纲，为您提供几个不同侧重点的“一句话简介”供选择：

**侧重“协同关系”**（最推荐，有画面感）：
> 本作围绕“Agent”展开，讲述其由**感知层**（洞察环境）、**决策层**（运筹帷幄）与**执行层**（落地行动）三部分协同运作，从而在复杂世界中完成使命的故事。

**侧重“技术逻辑”**（更硬核）：
> 这是一个关于“Agent”的传奇，它凭借**输入层**的敏锐、**处理层**的智慧与**输出层**的果决，三位一体地驱动着每一次行动与蜕变。

**侧重“成长叙事”**（拟人化）：
> 故事聚焦于“Agent”的觉醒，它通过**感官**（感知）、**思维**（决策）与**肢体**（执行）的完美融合，在一次次挑战中进化，最终成为掌控全局的关键存在。

**极简版**：
> 一个Agent的史诗，始于**感知**的洞察，成于**决策**的智慧，终于**执行**的魄力。


## 4. create_agent 的流式

Agent 内部有模型循环，`messages` 模式天然只流出**最终回答的 token**
（中间的工具调用消息 content 为空，被过滤掉）：

In [13]:

from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def get_weather(city: str) -> str:
    """查询城市天气。Args: city: 城市名"""
    return {"北京": "晴 25C"}.get(city, "未收录")

agent = create_agent(model=model, tools=[get_weather],
                     system_prompt="简洁回答，必须查工具。")

print("Agent 回答: ", end="")
for mode, chunk in agent.stream(
        {"messages": [{"role": "user", "content": "北京天气？"}]},
        stream_mode=["updates", "messages"]):
    if mode == "messages":
        msg, meta = chunk
        c = getattr(msg, "content", "")
        if c and meta.get("langgraph_node") == "model" and not getattr(msg, "tool_calls", None):
            print(c, end="", flush=True)
print()

Agent 回答: 北京：晴，25°C。


## 5. 自定义回调：token 与耗时统计

继承 `BaseCallbackHandler`，挂到任意 Runnable 上：
`on_llm_new_token` 逐 token 触发，`on_llm_end` 汇总——
自己写一个极简"计费表"：

In [ ]:

import time
from langchain_core.callbacks import BaseCallbackHandler

class Meter(BaseCallbackHandler):
    """统计 LLM 调用次数、token 数与总耗时"""
    def __init__(self):
        self.calls, self.tokens, self.t0 = 0, 0, time.time()

    def on_chat_model_start(self, serialized, messages, **kw):
        self.calls += 1

    def on_llm_new_token(self, token, **kw):
        self.tokens += 1

    def report(self):
        return (f"调用 {self.calls} 次 | ~{self.tokens} tokens | "
                f"耗时 {time.time() - self.t0:.1f}s")

meter = Meter()
r = model.invoke("用 30 字解释回调机制", config={"callbacks": [meter]})
print(r.content)
print("账单:", meter.report())

## 6. LangSmith 追踪（已配置）

项目根 `.env` 中 `LANGSMITH_TRACING=true` 已开启，**无需改代码**：
每次运行自动上报到 https://smith.langchain.com （项目名 `langchain_demo`）。

能看到：完整调用树、每步输入输出、token/费用、延迟瀑布图。
本地调试用 `stream_mode="debug"`，线上排障用 LangSmith，互为补充。

## 7. 常见问题（FAQ）

| 问题 | 原因 | 解决 |
|---|---|---|
| messages 模式流不出 token | 模型未真流式 / 过滤条件太严 | 确认走 stream()；meta 里看 langgraph_node |
| 输出夹杂空 content 块 | 工具调用消息 content 为空 | 加 `if c` 与 `not tool_calls` 过滤 |
| 回调没触发 | 挂错位置 | config={"callbacks": [...]} 传给 invoke/stream |
| updates 里出现 `__end__` | 正常现象 | 遍历时跳过该键 |
| LangSmith 看不到数据 | tracing 未开/网络不通 | 检查 .env 的 LANGSMITH_TRACING 与网络 |